In [ ]:
from pathlib import Path
import numpy as np 
import tifffile as tf
import pandas as pd
import skimage
from sklearn.preprocessing import MinMaxScaler
import pickle
import scipy.cluster.hierarchy as sch 
import matplotlib.pyplot as plt
from collections import Counter
import cv2
from matplotlib_scalebar.scalebar import ScaleBar
from pyFlowSOM import map_data_to_nodes, som
from alpineer import io_utils, misc_utils
from pyarrow.lib import ArrowInvalid
import seaborn as sns 
from tqdm import tqdm 

In [ ]:
pixel_clustering_path =  r"" # path to pixel clustering results

In [ ]:
all_channels = ['PODXL', 'LRP2', 'PANCK' 'CD20', 'CD68', 'HE_R', 'HE_G', 'HE_B', 'PAS_R', 'PAS_G', 'PAS_B', 'T_R', 'T_G', 'T_B']

In [ ]:
def compute_som_weights(pixels_subset_no_meta, prefix, xdim = 10, ydim = 10, num_passes = 1, lr_start=0.05, lr_end=0.01, seed = 42):
    data=pixels_subset_no_meta.values.astype(np.float64)
    som_weights = som(
            data=data, xdim=xdim, ydim=ydim,
            rlen=num_passes, alpha_range=(lr_start, lr_end), seed=seed
        )
    del data 
    som_weights = np.reshape(som_weights, (xdim * ydim, som_weights.shape[-1]))
    
    weights_df = pd.DataFrame(som_weights, columns=pixels_subset_no_meta.columns.values)
    del som_weights
    return weights_df

In [ ]:
def assign_som_clusters(weights_df, pixels_no_meta, batch_size): 
    cluster_labels = []

    
    for i in np.arange(0, pixels_no_meta.shape[0], batch_size):
        

        cluster_labels.append(map_data_to_nodes(
            weights_df.values.astype(np.float64),
            pixels_no_meta.loc[
                i:min(i + batch_size-1, pixels_no_meta.shape[0]), pixels_no_meta.columns.values
            ].values.astype(np.float64)
        )[0])

    pixels_no_meta['pixel_som_cluster'] = np.concatenate(cluster_labels)
    return pixels_no_meta
    
    

In [ ]:
def cluster (pixels, prefix, batch_size, drop_position, weights_df = None, xdim = None, ydim= None):
    if(drop_position):
        pixels_no_meta = pixels.drop(columns=['X', 'Y'])
    else:
        pixels_no_meta = pixels

    if(weights_df is None):
        if(xdim is None and ydim is None):
            weights_df = compute_som_weights(pixels_no_meta, prefix)
        else:
            weights_df = compute_som_weights(pixels_no_meta, prefix, xdim=xdim, ydim=ydim)

    pixels_no_meta_som_clusters = assign_som_clusters(weights_df, pixels_no_meta, batch_size)
   

    return pixels_no_meta_som_clusters, weights_df 

In [ ]:
def compute_quantization_cost_optimized(weights_df, non_empty_pixels_som_clusters_no_meta, all_channels):
    grouped = non_empty_pixels_som_clusters_no_meta.groupby('pixel_som_cluster')
    err = 0
    num_clusters = len(grouped)

    for cluster, group in grouped:
        data_cluster_values = group[all_channels].values
        weight_cluster_values = weights_df.iloc[cluster - 1][all_channels].values
        err += np.linalg.norm(data_cluster_values - weight_cluster_values, axis=1).sum()

    return err / num_clusters


In [ ]:
non_empty_pixels_scaled = pd.read_csv(pixel_clustering_path + '/non_empty_pixels_scaled.csv').drop(columns=['Unnamed: 0'])

In [ ]:
quantization_cost_dim = ((1, 1), (2, 2), (3, 3), (4, 4), (5, 5), (6, 6), (7, 7), (8, 8), (9, 9), (10, 10))

In [ ]:
quantization_cost = []

In [ ]:
for d in tqdm(quantization_cost_dim): 
    non_empty_pixels_som_clusters_no_meta, weights_df =  cluster(non_empty_pixels_scaled, '1x_' + str(d[0]*d[1]), 100000000, drop_position=True, weights_df=None, xdim=d[0], ydim=d[1])
    quantization_cost.append(compute_quantization_cost_optimized(weights_df, non_empty_pixels_som_clusters_no_meta, all_channels))
    del non_empty_pixels_som_clusters_no_meta, weights_df
    

In [ ]:
with open(pixel_clustering_path + '/quantization_cost.pickle', 'wb') as f: 
    pickle.dump(quantization_cost, f)

In [ ]:
print(quantization_cost[1]/quantization_cost[2])

In [ ]:
print(quantization_cost[2]/quantization_cost[3])

In [ ]:
plt.rcParams['font.size'] = 20

fig, ax = plt.subplots(1)
fig.set_figheight(8)
fig.set_figwidth(20)
ax.plot([(i)**2 for i in range(1, 11)], quantization_cost, label='Quantization Cost')
for x, y in zip([(i)**2 for i in range(1, 11)], quantization_cost): 
        ax.annotate(str(x), (x, y), 
                     textcoords="offset points",
                     xytext=(0, 0),
                     ha='center') 
ax.set_xlabel('Number of Clusters')
ax.set_ylabel('Quantization Cost')


plt.savefig(pixel_clustering_path + '/cluster_optimization.png')